In [109]:
# Step 1: Import required libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0  # Changed from VGG16
from tensorflow.keras.applications.efficientnet import preprocess_input  # Critical: Different preprocessing
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout  # Replaced Flatten with GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

In [110]:
# Step 2: Set paths and parameters
train_dir = "dataset/train"
valid_dir = "dataset/valid"

IMG_SIZE = (224, 224)   # EfficientNetB0 standard input size. Can also use (240, 240) or (256, 256) for potentially better performance.
BATCH_SIZE = 32
EPOCHS = 10

In [111]:
# Step 3 (CORRECTED): Create data generators with EfficientNet-specific preprocessing

# Let's test the preprocess_input function first
test_image = np.ones((224, 224, 3)) * 128.0  # Create an image with value 128
processed_test = preprocess_input(test_image)
print(f"Preprocess_input test: [128, 128, 128] -> {processed_test[0, 0, :]}")

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # This should scale pixels appropriately
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input  # Apply same preprocessing to validation
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='rgb'
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# Let's verify the preprocessing is working by checking a batch
sample_batch, sample_labels = next(train_generator)
print(f"\nAfter preprocessing - Sample batch range: [{sample_batch.min()}, {sample_batch.max()}]")

Preprocess_input test: [128, 128, 128] -> [128. 128. 128.]
Found 2540 images belonging to 2 classes.
Found 100 images belonging to 2 classes.

After preprocessing - Sample batch range: [0.0, 255.0]


In [112]:
# Step 4 (Final Working Solution): Build without weights, then load weights manually with skip_mismatch

# 1. First build the model WITHOUT any weights
base_model = EfficientNetB0(weights=None, include_top=False, input_shape=(224, 224, 3))

# 2. Now try to load Imagenet weights, but skip any layers that have shape mismatches
# This will allow the rest of the model to load pre-trained weights
try:
    base_model.load_weights(tf.keras.utils.get_file(
        'efficientnetb0_notop.h5',
        'https://storage.googleapis.com/keras-applications/efficientnetb0_notop.h5',
        cache_subdir='models'
    ), by_name=True, skip_mismatch=True)
    print("✓ Weights loaded successfully with skip_mismatch=True")
except Exception as e:
    print(f"Could not load weights: {e}")
    print("Continuing with randomly initialized weights")

# 3. Freeze all convolutional layers
for layer in base_model.layers:
    layer.trainable = False

print("Model ready for training!")
print(f"Model input shape: {base_model.input_shape}")
print(f"Model output shape: {base_model.output_shape}")

# 4. Let's test that it works with our data
test_batch, _ = next(train_generator)
output = base_model(test_batch)
print(f"Test forward pass output shape: {output.shape}")

✓ Weights loaded successfully with skip_mismatch=True
Model ready for training!
Model input shape: (None, 224, 224, 3)
Model output shape: (None, 7, 7, 1280)
Test forward pass output shape: (32, 7, 7, 1280)


In [113]:
# Step 5: Add custom classification layers for EfficientNetB0
x = GlobalAveragePooling2D()(base_model.output)  # Changed from Flatten()
x = Dense(128, activation='relu')(x)  # Reduced from 256 to 128 (EfficientNet features are richer)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)  # binary classification

model = Model(inputs=base_model.input, outputs=output)

In [114]:
# Step 6: Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_29      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_42        │ (None, 224, 224,  │          0 │ input_layer_29[0… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_30    │ (None, 224, 224,  │          7 │ rescaling_42[0][… │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ normalization_30… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 112, 112,  │         64 │ block1a_project_

 Total params: 4,213,668 (16.07 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [115]:
# Step 7: Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,  # Added for clarity
    epochs=EPOCHS,
    validation_data=valid_generator,
    validation_steps=valid_generator.samples // BATCH_SIZE   # Added for clarity
)

/Users/user89/Desktop/ASD_v/venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


Python(21512,0x1732a3000) malloc: Failed to allocate segment from range group - out of space


79/79 ━━━━━━━━━━━━━━━━━━━━ 26s 231ms/step - accuracy: 0.6128 - loss: 0.6581 - val_accuracy: 0.7292 - val_loss: 0.5873
Epoch 2/10
 1/79 ━━━━━━━━━━━━━━━━━━━━ 12s 167ms/step - accuracy: 0.5625 - loss: 0.6505

/Users/user89/Desktop/ASD_v/venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5625 - loss: 0.6505 - val_accuracy: 0.7292 - val_loss: 0.5861
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 163ms/step - accuracy: 0.6974 - loss: 0.5755 - val_accuracy: 0.6875 - val_loss: 0.5515
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6875 - loss: 0.5570 - val_accuracy: 0.6979 - val_loss: 0.5504
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 161ms/step - accuracy: 0.7201 - loss: 0.5530 - val_accuracy: 0.7396 - val_loss: 0.5276
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9062 - loss: 0.4359 - val_accuracy: 0.7396 - val_loss: 0.5265
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 162ms/step - accuracy: 0.7225 - loss: 0.5553 - val_accuracy: 0.7396 - val_loss: 0.5096
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6562 - loss: 0.5663 - val_accuracy: 0.7396 - val_loss: 0.5090
Epoch 9/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 162ms/step - accuracy: 0.7285 - loss: 0.5396 - val_accuracy: 0.7396 - val_loss

In [116]:
# Step 8: Evaluate the model
loss, accuracy = model.evaluate(valid_generator)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

3/4 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.6163 - loss: 0.6681

Python(21512,0x1735eb000) malloc: Failed to allocate segment from range group - out of space


4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 473ms/step - accuracy: 0.7500 - loss: 0.5072
Validation Loss: 0.5072
Validation Accuracy: 0.7500


In [117]:
# Step 9: Get predictions for evaluation
# Note: Make sure to reset the generator to ensure proper alignment between predictions and true labels
valid_generator.reset()  # Added this line to reset generator state
y_pred = model.predict(valid_generator)
y_pred_classes = (y_pred > 0.5).astype(int)

print(classification_report(valid_generator.classes, y_pred_classes, target_names=['Autistic', 'Non_Autistic']))

cm = confusion_matrix(valid_generator.classes, y_pred_classes)
print("Confusion Matrix:\n", cm)

4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step
              precision    recall  f1-score   support

    Autistic       0.88      0.58      0.70        50
Non_Autistic       0.69      0.92      0.79        50

    accuracy                           0.75       100
   macro avg       0.78      0.75      0.74       100
weighted avg       0.78      0.75      0.74       100

Confusion Matrix:
 [[29 21]
 [ 4 46]]


In [118]:
model.save("autism_detection_efficientnetb0.h5")
print("Model saved as: autism_detection_efficientnetb0.h5")

Model saved as: autism_detection_efficientnetb0.h5


In [119]:
import numpy as np
from tensorflow.keras.preprocessing import image
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input  # Added for EfficientNet preprocessing

# Load your trained EfficientNetB0 model
model = tf.keras.models.load_model("autism_detection_efficientnetb0.h5")  # Updated filename

# Function to predict autism or not
def predict_autism(img_path):
    IMG_SIZE = (224, 224)  # Same as training
    img = image.load_img(img_path, target_size=IMG_SIZE, color_mode='rgb')  # Ensure RGB mode
    img_array = image.img_to_array(img)
    img_array = preprocess_input(img_array)  # Changed from /255.0 to EfficientNet preprocessing
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    prediction = model.predict(img_array)[0][0]

    if prediction > 0.5:
        print(f"{img_path} --> Non_Autistic ({prediction:.4f})")
    else:
        print(f"{img_path} --> Autistic ({prediction:.4f})")

# Example usage:
test_image_path = "dataset/train/non_autistic/Non_Autistic.1002.jpg"
predict_autism(test_image_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step
dataset/train/non_autistic/Non_Autistic.1002.jpg --> Non_Autistic (0.9289)


In [120]:
import tensorflow as tf

# Load trained EfficientNetB0 model
model = tf.keras.models.load_model("autism_detection_efficientnetb0.h5")

# EfficientNetB0 has 7 main blocks + top layers
# Let's unfreeze the last 2 blocks and the top layers for fine-tuning
trainable_prefixes = ['block6', 'block7', 'top_']

for layer in model.layers:
    # Check if layer belongs to the last blocks or top layers
    if any(layer.name.startswith(prefix) for prefix in trainable_prefixes):
        layer.trainable = True
        print(f"Unfrozen: {layer.name}")
    else:
        layer.trainable = False

print("\nSummary of trainable layers:")
trainable_count = sum(layer.trainable for layer in model.layers)
total_count = len(model.layers)
print(f"Trainable layers: {trainable_count}/{total_count}")

Unfrozen: block6a_expand_conv
Unfrozen: block6a_expand_bn
Unfrozen: block6a_expand_activation
Unfrozen: block6a_dwconv_pad
Unfrozen: block6a_dwconv
Unfrozen: block6a_bn
Unfrozen: block6a_activation
Unfrozen: block6a_se_squeeze
Unfrozen: block6a_se_reshape
Unfrozen: block6a_se_reduce
Unfrozen: block6a_se_expand
Unfrozen: block6a_se_excite
Unfrozen: block6a_project_conv
Unfrozen: block6a_project_bn
Unfrozen: block6b_expand_conv
Unfrozen: block6b_expand_bn
Unfrozen: block6b_expand_activation
Unfrozen: block6b_dwconv
Unfrozen: block6b_bn
Unfrozen: block6b_activation
Unfrozen: block6b_se_squeeze
Unfrozen: block6b_se_reshape
Unfrozen: block6b_se_reduce
Unfrozen: block6b_se_expand
Unfrozen: block6b_se_excite
Unfrozen: block6b_project_conv
Unfrozen: block6b_project_bn
Unfrozen: block6b_drop
Unfrozen: block6b_add
Unfrozen: block6c_expand_conv
Unfrozen: block6c_expand_bn
Unfrozen: block6c_expand_activation
Unfrozen: block6c_dwconv
Unfrozen: block6c_bn
Unfrozen: block6c_activation
Unfrozen: block

In [121]:
from tensorflow.keras.optimizers import Adam

# Recompile with a lower learning rate to avoid damaging pretrained weights
model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Fine-tuning uses very small LR
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_29      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_42        │ (None, 224, 224,  │          0 │ input_layer_29[0… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_30    │ (None, 224, 224,  │          7 │ rescaling_42[0][… │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ normalization_30… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 112, 112,  │         64 │ block1a_project_

 Total params: 4,213,668 (16.07 MB)

 Trainable params: 3,155,740 (12.04 MB)

 Non-trainable params: 1,057,928 (4.04 MB)

In [122]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input  # Import EfficientNet preprocessing

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Use EfficientNet-specific preprocessing instead of rescale=1./255
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # Changed from rescale=1./255
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input  # Changed from rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    "dataset/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='rgb'  # Ensure RGB format
)

valid_generator = valid_datagen.flow_from_directory(
    "dataset/valid",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
    color_mode='rgb'  # Ensure RGB format
)

Found 2540 images belonging to 2 classes.
Found 100 images belonging to 2 classes.


In [131]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,                     # stop if no improvement for 7 epochs
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,                     # reduce LR by 5x
    patience=3,                     # wait 3 epochs before reducing
    min_lr=1e-6
)

checkpoint = ModelCheckpoint(
    "best_efficientnetb0.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

# Train (fine-tuning EfficientNetB0)
history_finetune = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=50,                       # set high, early stopping will cut it
    initial_epoch=0,
    validation_data=valid_generator,
    validation_steps=len(valid_generator),
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)


Epoch 1/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7339 - loss: 0.5582
Epoch 1: val_accuracy improved from None to 0.56000, saving model to best_efficientnetb0.h5


80/80 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.7445 - loss: 0.5423 - val_accuracy: 0.5600 - val_loss: 2.5236 - learning_rate: 0.0010
Epoch 2/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7961 - loss: 0.4514
Epoch 2: val_accuracy improved from 0.56000 to 0.59000, saving model to best_efficientnetb0.h5


80/80 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.7870 - loss: 0.4693 - val_accuracy: 0.5900 - val_loss: 1.1744 - learning_rate: 0.0010
Epoch 3/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 17s/step - accuracy: 0.7730 - loss: 0.4893 
Epoch 3: val_accuracy improved from 0.59000 to 0.73000, saving model to best_efficientnetb0.h5


80/80 ━━━━━━━━━━━━━━━━━━━━ 1341s 17s/step - accuracy: 0.7752 - loss: 0.4833 - val_accuracy: 0.7300 - val_loss: 1.0756 - learning_rate: 0.0010
Epoch 4/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8090 - loss: 0.4230
Epoch 4: val_accuracy did not improve from 0.73000
80/80 ━━━━━━━━━━━━━━━━━━━━ 158s 2s/step - accuracy: 0.7929 - loss: 0.4564 - val_accuracy: 0.7200 - val_loss: 0.7986 - learning_rate: 0.0010
Epoch 5/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 823ms/step - accuracy: 0.8160 - loss: 0.4023
Epoch 5: val_accuracy improved from 0.73000 to 0.80000, saving model to best_efficientnetb0.h5


80/80 ━━━━━━━━━━━━━━━━━━━━ 67s 839ms/step - accuracy: 0.8193 - loss: 0.4115 - val_accuracy: 0.8000 - val_loss: 0.5077 - learning_rate: 0.0010
Epoch 6/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 882ms/step - accuracy: 0.8109 - loss: 0.4216
Epoch 6: val_accuracy improved from 0.80000 to 0.81000, saving model to best_efficientnetb0.h5


80/80 ━━━━━━━━━━━━━━━━━━━━ 72s 896ms/step - accuracy: 0.8142 - loss: 0.4207 - val_accuracy: 0.8100 - val_loss: 0.4996 - learning_rate: 0.0010
Epoch 7/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 889ms/step - accuracy: 0.8142 - loss: 0.4298
Epoch 7: val_accuracy did not improve from 0.81000
80/80 ━━━━━━━━━━━━━━━━━━━━ 73s 899ms/step - accuracy: 0.8098 - loss: 0.4382 - val_accuracy: 0.7500 - val_loss: 0.6169 - learning_rate: 0.0010
Epoch 8/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 839ms/step - accuracy: 0.7946 - loss: 0.4440
Epoch 8: val_accuracy did not improve from 0.81000
80/80 ━━━━━━━━━━━━━━━━━━━━ 68s 847ms/step - accuracy: 0.7906 - loss: 0.4448 - val_accuracy: 0.7300 - val_loss: 0.5950 - learning_rate: 0.0010
Epoch 9/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 981ms/step - accuracy: 0.8002 - loss: 0.4205
Epoch 9: val_accuracy did not improve from 0.81000
80/80 ━━━━━━━━━━━━━━━━━━━━ 79s 991ms/step - accuracy: 0.8106 - loss: 0.4168 - val_accuracy: 0.8100 - val_loss: 0.4505 - learning_rate: 0.0010
Epoch 10/50
80/80 ━━━

80/80 ━━━━━━━━━━━━━━━━━━━━ 69s 864ms/step - accuracy: 0.8705 - loss: 0.3085 - val_accuracy: 0.8300 - val_loss: 0.4068 - learning_rate: 2.0000e-04
Epoch 19/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 864ms/step - accuracy: 0.8609 - loss: 0.3002
Epoch 19: val_accuracy did not improve from 0.83000
80/80 ━━━━━━━━━━━━━━━━━━━━ 70s 871ms/step - accuracy: 0.8685 - loss: 0.3041 - val_accuracy: 0.7800 - val_loss: 0.4070 - learning_rate: 2.0000e-04
Epoch 20/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 840ms/step - accuracy: 0.8842 - loss: 0.2665
Epoch 20: val_accuracy did not improve from 0.83000
80/80 ━━━━━━━━━━━━━━━━━━━━ 68s 848ms/step - accuracy: 0.8791 - loss: 0.2784 - val_accuracy: 0.8100 - val_loss: 0.4068 - learning_rate: 2.0000e-04
Epoch 21/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8825 - loss: 0.2945
Epoch 21: val_accuracy did not improve from 0.83000
80/80 ━━━━━━━━━━━━━━━━━━━━ 95s 1s/step - accuracy: 0.8823 - loss: 0.2873 - val_accuracy: 0.7800 - val_loss: 0.4053 - learning_rate: 2.0000e-04
Epoch

In [138]:
loss, accuracy = model.evaluate(valid_generator)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.7800 - loss: 0.4053
Validation Loss: 0.4053
Validation Accuracy: 0.7800


In [133]:
# Save only the weights (use the exact required format)
model.save_weights("autism_detection_efficientnetb0_finetuned.weights.h5")
print("Model weights saved successfully to autism_detection_efficientnetb0_finetuned.weights.h5")

Model weights saved successfully to autism_detection_efficientnetb0_finetuned.weights.h5


In [134]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Get predictions
y_true = valid_generator.classes
y_pred_prob = model.predict(valid_generator)
y_pred = (y_pred_prob > 0.5).astype(int).reshape(-1)

# Labels
labels = list(valid_generator.class_indices.keys())

# Report
print(classification_report(y_true, y_pred, target_names=labels))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

4/4 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step
              precision    recall  f1-score   support

    Autistic       0.82      0.72      0.77        50
Non_Autistic       0.75      0.84      0.79        50

    accuracy                           0.78       100
   macro avg       0.78      0.78      0.78       100
weighted avg       0.78      0.78      0.78       100

Confusion Matrix:
[[36 14]
 [ 8 42]]


In [143]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

# 1. First, reconstruct the EXACT model architecture used during training
def create_efficientnet_model():
    # Create base model (same as during training)
    base_model = tf.keras.applications.EfficientNetB0(
        weights=None, 
        include_top=False, 
        input_shape=(224, 224, 3)
    )
    
    # Add the exact same custom layers you used during training
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)  # Same number of units
    x = Dropout(0.5)(x)                   # Same dropout rate
    output = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=base_model.input, outputs=output)
    return model

# 2. Create the model architecture
print("Creating model architecture...")
model = create_efficientnet_model()

# 3. Load the fine-tuned weights
print("Loading fine-tuned weights...")
model.load_weights("autism_detection_efficientnetb0_finetuned.weights.h5")  # Use the correct weights file
print("✓ Model loaded successfully!")

# 4. Compile the model (optional but good practice)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 5. Prediction function
def predict_autism(img_path):
    IMG_SIZE = (224, 224)
    
    # Load and preprocess image
    img = image.load_img(img_path, target_size=IMG_SIZE, color_mode='rgb')
    img_array = image.img_to_array(img)
    img_array = preprocess_input(img_array)  # EfficientNet-specific preprocessing
    img_array = np.expand_dims(img_array, axis=0)  # add batch dimension

    # Make prediction
    prediction = model.predict(img_array, verbose=0)[0][0]
    
    # Interpret results
    if prediction > 0.50:
        print(f"🟦 Prediction: Non_Autistic ({prediction:.4f} confidence)")
        return "Non_Autistic", prediction
    else:
        print(f"🟥 Prediction: Autistic ({1 - prediction:.4f} confidence)")
        return "Autistic", 1 - prediction

# 6. Example usage
print("\nMaking prediction...")
result, confidence = predict_autism("dataset/test/non_autistic/Non_Autistic.24 11.44.29.jpg")
print(f"Final result: {result} with {confidence:.1%} confidence")

Creating model architecture...
Loading fine-tuned weights...
✓ Model loaded successfully!

Making prediction...
🟦 Prediction: Non_Autistic (0.7556 confidence)
Final result: Non_Autistic with 75.6% confidence


In [144]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os

# 1. Reconstruct the model architecture
def create_efficientnet_model():
    base_model = tf.keras.applications.EfficientNetB0(
        weights=None, 
        include_top=False, 
        input_shape=(224, 224, 3)
    )
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=base_model.input, outputs=output)
    return model

# 2. Create and load the model
print("Creating model architecture...")
model = create_efficientnet_model()
print("Loading fine-tuned weights...")
model.load_weights("autism_detection_efficientnetb0_finetuned.weights.h5")
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("✓ Model loaded successfully!")

# 3. Function to evaluate model on test directory
def evaluate_model(test_dir="dataset/test"):
    """
    Evaluate the model on the test dataset
    """
    print(f"🔍 Evaluating model on: {test_dir}")
    
    y_true = []
    y_pred_probs = []
    y_pred_classes = []
    processed_images = 0
    
    # Define class folders based on your sample path
    class_folders = {
        'autistic': 'autistic',
        'non_autistic': 'non_autistic'
    }
    
    # Process each class
    for class_label, folder_name in class_folders.items():
        class_dir = os.path.join(test_dir, folder_name)
        true_label = 0 if class_label == 'autistic' else 1
        
        if not os.path.exists(class_dir):
            print(f"⚠️  Directory not found: {class_dir}")
            continue
            
        print(f"📂 Processing {class_label} images from: {class_dir}")
        
        # Process each image in the class directory
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(class_dir, img_name)
                
                try:
                    # Load and preprocess image
                    img = image.load_img(img_path, target_size=(224, 224), color_mode='rgb')
                    img_array = image.img_to_array(img)
                    img_array = preprocess_input(img_array)
                    img_array = np.expand_dims(img_array, axis=0)
                    
                    # Predict
                    prediction = model.predict(img_array, verbose=0)[0][0]
                    
                    y_true.append(true_label)
                    y_pred_probs.append(prediction)
                    y_pred_classes.append(1 if prediction > 0.5 else 0)
                    
                    processed_images += 1
                    
                    # Print progress every 10 images
                    if processed_images % 10 == 0:
                        print(f"   Processed {processed_images} images...")
                        
                except Exception as e:
                    print(f"❌ Error processing {img_path}: {e}")
    
    if processed_images == 0:
        print("❌ No images found for evaluation!")
        return 0
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred_classes)
    
    print(f"\n{'='*50}")
    print("📊 EVALUATION RESULTS")
    print(f"{'='*50}")
    print(f"✅ Total images evaluated: {processed_images}")
    print(f"🎯 Overall Accuracy: {accuracy:.4f} ({accuracy:.2%})")
    
    print(f"\n📋 Classification Report:")
    print(classification_report(y_true, y_pred_classes, 
                               target_names=['Autistic', 'Non_Autistic'],
                               digits=4))
    
    print(f"\n📊 Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred_classes)
    print(cm)
    
    # Detailed analysis
    print(f"\n🔍 Detailed Analysis:")
    print(f"True Positives (Autistic correctly identified): {cm[0][0]}")
    print(f"False Negatives (Autistic misclassified): {cm[0][1]}")
    print(f"False Positives (Non-Autistic misclassified): {cm[1][0]}")
    print(f"True Negatives (Non-Autistic correctly identified): {cm[1][1]}")
    
    # Class-wise accuracy
    autistic_indices = [i for i, label in enumerate(y_true) if label == 0]
    non_autistic_indices = [i for i, label in enumerate(y_true) if label == 1]
    
    if autistic_indices:
        autistic_accuracy = accuracy_score([y_true[i] for i in autistic_indices], 
                                          [y_pred_classes[i] for i in autistic_indices])
        print(f"🎯 Autistic class accuracy: {autistic_accuracy:.4f} ({autistic_accuracy:.2%})")
    
    if non_autistic_indices:
        non_autistic_accuracy = accuracy_score([y_true[i] for i in non_autistic_indices], 
                                              [y_pred_classes[i] for i in non_autistic_indices])
        print(f"🎯 Non-Autistic class accuracy: {non_autistic_accuracy:.4f} ({non_autistic_accuracy:.2%})")
    
    return accuracy

# 4. Single image prediction function (for testing)
def predict_single_image(img_path):
    """
    Predict a single image and return detailed results
    """
    print(f"\n🔮 Predicting single image: {img_path}")
    
    try:
        # Load and preprocess image
        img = image.load_img(img_path, target_size=(224, 224), color_mode='rgb')
        img_array = image.img_to_array(img)
        img_array = preprocess_input(img_array)
        img_array = np.expand_dims(img_array, axis=0)
        
        # Predict
        prediction = model.predict(img_array, verbose=0)[0][0]
        
        # Determine actual class from path
        actual_class = "Unknown"
        if "non_autistic" in img_path.lower():
            actual_class = "Non_Autistic"
        elif "autistic" in img_path.lower():
            actual_class = "Autistic"
        
        # Interpret results
        if prediction > 0.5:
            predicted_class = "Non_Autistic"
            confidence = prediction
        else:
            predicted_class = "Autistic"
            confidence = 1 - prediction
        
        print(f"📁 Image: {os.path.basename(img_path)}")
        print(f"✅ Actual class: {actual_class}")
        print(f"🎯 Predicted: {predicted_class}")
        print(f"📈 Confidence: {confidence:.4f} ({confidence:.2%})")
        print(f"🔢 Raw prediction score: {prediction:.6f}")
        
        return predicted_class, confidence
        
    except Exception as e:
        print(f"❌ Error processing image: {e}")
        return None, None

# 5. Main execution
if __name__ == "__main__":
    # Test with your sample image first
    sample_image_path = "dataset/test/non_autistic/Non_Autistic.24 11.44.29.jpg"
    
    if os.path.exists(sample_image_path):
        print("🧪 Testing with sample image...")
        predict_single_image(sample_image_path)
        print("\n" + "="*50 + "\n")
    else:
        print(f"⚠️  Sample image not found: {sample_image_path}")
        print("Please check the file path and try again.")
    
    # Evaluate on entire test dataset
    test_directory = "dataset/test"  # Change this if your test data is elsewhere
    if os.path.exists(test_directory):
        final_accuracy = evaluate_model(test_directory)
        print(f"\n🎉 FINAL MODEL ACCURACY: {final_accuracy:.2%}")
    else:
        print(f"❌ Test directory not found: {test_directory}")
        print("Please make sure your test dataset is in the correct location.")

Creating model architecture...
Loading fine-tuned weights...
✓ Model loaded successfully!
🧪 Testing with sample image...

🔮 Predicting single image: dataset/test/non_autistic/Non_Autistic.24 11.44.29.jpg
📁 Image: Non_Autistic.24 11.44.29.jpg
✅ Actual class: Non_Autistic
🎯 Predicted: Non_Autistic
📈 Confidence: 0.7556 (75.56%)
🔢 Raw prediction score: 0.755609


🔍 Evaluating model on: dataset/test
📂 Processing autistic images from: dataset/test/autistic
   Processed 10 images...
   Processed 20 images...
   Processed 30 images...
   Processed 40 images...
   Processed 50 images...
   Processed 60 images...
   Processed 70 images...
   Processed 80 images...
   Processed 90 images...
   Processed 100 images...
   Processed 110 images...
   Processed 120 images...
   Processed 130 images...
   Processed 140 images...
   Processed 150 images...
📂 Processing non_autistic images from: dataset/test/non_autistic
   Processed 160 images...
   Processed 170 images...
   Processed 180 images...
   